# 🗄️ NYC Taxi SQL Analytics Case Study
**Author:** Tharun Kumar Reddy Byreddy

**Goal:** Extract business insights from NYC Taxi trip data using advanced SQL techniques and Python visualizations.

---

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
print('Libraries loaded!')

## 2. Generate Synthetic NYC Taxi Data

In [ ]:
np.random.seed(42)
n = 50000

# Simulate realistic trip patterns
hours = np.random.choice(
    range(24),
    size=n,
    p=[0.02,0.01,0.01,0.01,0.01,0.02,
       0.04,0.06,0.07,0.05,0.04,0.05,
       0.05,0.04,0.04,0.04,0.05,0.06,
       0.07,0.07,0.06,0.05,0.04,0.03]
)

base_dates = pd.date_range('2023-01-01', periods=365)
dates = np.random.choice(base_dates, n)
pickup_datetime = pd.to_datetime(dates) + pd.to_timedelta(hours, unit='h')

trip_distance = np.random.exponential(3, n).clip(0.1, 30)
fare_amount   = 2.5 + trip_distance * 2.5 + np.random.normal(0, 2, n)
fare_amount   = fare_amount.clip(2.5, 150)
tip_amount    = np.where(
    np.random.random(n) > 0.3,
    fare_amount * np.random.uniform(0.1, 0.35, n),
    0
)

df = pd.DataFrame({
    'pickup_datetime':    pickup_datetime,
    'trip_distance':      trip_distance.round(2),
    'fare_amount':        fare_amount.round(2),
    'tip_amount':         tip_amount.round(2),
    'total_amount':       (fare_amount + tip_amount).round(2),
    'passenger_count':    np.random.choice([1,2,3,4,5,6], n,
                              p=[0.6,0.2,0.1,0.05,0.03,0.02]),
    'payment_type':       np.random.choice(
                              ['Credit Card','Cash','No Charge','Dispute'],
                              n, p=[0.67,0.30,0.02,0.01]),
    'vendor_id':          np.random.choice(['Vendor A','Vendor B'], n),
})

df['hour']        = df['pickup_datetime'].dt.hour
df['day_of_week'] = df['pickup_datetime'].dt.day_name()
df['month']       = df['pickup_datetime'].dt.month
df['date']        = df['pickup_datetime'].dt.date

print('Dataset shape:', df.shape)
print('Date range:', df['pickup_datetime'].min(), 'to', df['pickup_datetime'].max())
df.head()

## 3. Basic Analysis

In [ ]:
print('=== OVERALL SUMMARY ===')
print(f'Total Trips    : {len(df):,}')
print(f'Total Revenue  : ${df["fare_amount"].sum():,.2f}')
print(f'Avg Fare       : ${df["fare_amount"].mean():.2f}')
print(f'Avg Tip        : ${df["tip_amount"].mean():.2f}')
print(f'Avg Distance   : {df["trip_distance"].mean():.2f} miles')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['payment_type'].value_counts().plot(
    kind='bar', ax=axes[0], color='steelblue', edgecolor='white'
)
axes[0].set_title('Trips by Payment Type')
axes[0].tick_params(rotation=30)

df['passenger_count'].value_counts().sort_index().plot(
    kind='bar', ax=axes[1], color='coral', edgecolor='white'
)
axes[1].set_title('Trips by Passenger Count')
axes[1].tick_params(rotation=0)

plt.tight_layout()
plt.savefig('../results/basic_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Revenue Trends

In [ ]:
hourly = df.groupby('hour').agg(
    total_trips=('fare_amount','count'),
    total_revenue=('fare_amount','sum'),
    avg_fare=('fare_amount','mean')
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(hourly['hour'], hourly['total_trips'],
            color='steelblue', edgecolor='white')
axes[0].set_title('Trip Count by Hour of Day')
axes[0].set_xlabel('Hour')
axes[0].set_ylabel('Total Trips')

axes[1].bar(hourly['hour'], hourly['total_revenue'],
            color='coral', edgecolor='white')
axes[1].set_title('Revenue by Hour of Day')
axes[1].set_xlabel('Hour')
axes[1].set_ylabel('Total Revenue ($)')

plt.tight_layout()
plt.savefig('../results/revenue_trends.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Window Functions — Rolling Average

In [ ]:
daily = df.groupby('date').agg(
    total_revenue=('fare_amount','sum'),
    total_trips=('fare_amount','count')
).reset_index()

daily['rolling_7day'] = daily['total_revenue'].rolling(7).mean()
daily['rolling_30day'] = daily['total_revenue'].rolling(30).mean()
daily['date'] = pd.to_datetime(daily['date'])

plt.figure(figsize=(14, 5))
plt.plot(daily['date'], daily['total_revenue'],
         alpha=0.3, color='steelblue', label='Daily Revenue')
plt.plot(daily['date'], daily['rolling_7day'],
         color='coral', linewidth=2, label='7-Day Rolling Avg')
plt.plot(daily['date'], daily['rolling_30day'],
         color='green', linewidth=2, label='30-Day Rolling Avg')
plt.title('Daily Revenue with Rolling Averages')
plt.xlabel('Date')
plt.ylabel('Revenue ($)')
plt.legend()
plt.tight_layout()
plt.savefig('../results/rolling_averages.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Tip Behavior Analysis

In [ ]:
df_tips = df[df['fare_amount'] > 0].copy()
df_tips['tip_pct'] = df_tips['tip_amount'] / df_tips['fare_amount'] * 100
df_tips['tip_category'] = pd.cut(
    df_tips['tip_pct'],
    bins=[-1, 0, 10, 20, 30, 100],
    labels=['No Tip','<10%','10-20%','20-30%','30%+']
)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

tip_counts = df_tips['tip_category'].value_counts().sort_index()
tip_counts.plot(kind='bar', ax=axes[0],
                color='steelblue', edgecolor='white')
axes[0].set_title('Tip Category Distribution')
axes[0].tick_params(rotation=30)

axes[1].scatter(df_tips['fare_amount'].sample(2000),
                df_tips['tip_amount'].sample(2000),
                alpha=0.3, color='coral', s=10)
axes[1].set_title('Fare vs Tip Amount')
axes[1].set_xlabel('Fare Amount ($)')
axes[1].set_ylabel('Tip Amount ($)')

plt.tight_layout()
plt.savefig('../results/tip_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Key Business Insights

In [ ]:
peak_hour = hourly.loc[hourly['total_trips'].idxmax(), 'hour']
peak_revenue_hour = hourly.loc[hourly['total_revenue'].idxmax(), 'hour']
top_payment = df['payment_type'].value_counts().index[0]
tip_rate = (df['tip_amount'] > 0).mean() * 100

print('='*55)
print('         KEY BUSINESS INSIGHTS')
print('='*55)
print(f'Total Trips Analyzed   : {len(df):,}')
print(f'Total Revenue          : ${df["fare_amount"].sum():,.0f}')
print(f'Peak Trip Hour         : {peak_hour}:00')
print(f'Peak Revenue Hour      : {peak_revenue_hour}:00')
print(f'Top Payment Method     : {top_payment}')
print(f'Tip Rate               : {tip_rate:.1f}%')
print(f'Avg Fare per Mile      : ${(df["fare_amount"]/df["trip_distance"]).mean():.2f}')
print('='*55)
print('Author: Tharun Kumar Reddy Byreddy')
print('M.S. Statistical Data Science | SFSU')